<a href="https://colab.research.google.com/github/Serx17/compliance-checker-230fz/blob/main/notebooks/03_final_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 📦 Установка зависимостей (обязательно при каждом запуске Colab)
!pip install chromadb sentence-transformers --quiet --disable-pip-version-check 2>/dev/null

import chromadb
from sentence_transformers import SentenceTransformer
import json
print("✅ Библиотеки загружены. Можно запускать основной код.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
✅ Библиотеки загружены. Можно запускать основной код.


In [7]:
# ==========================================
# 📦 1. УСТАНОВКА И ИМПОРТЫ (Для новой сессии)
# ==========================================
!pip install chromadb sentence-transformers --quiet --disable-pip-version-check 2>/dev/null

import chromadb
from sentence_transformers import SentenceTransformer
import json
import os
import shutil

print("✅ Библиотеки загружены.")

# ==========================================
# 🗄 2. НАСТРОЙКИ И ДАННЫЕ
# ==========================================
DB_PATH = "/content/chroma_db_230fz_v2"
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# Те же чанки, что и раньше (с обогащением)
LAW_CHUNKS = [
    {"id": "chunk_1", "source": "Статья 6, ч. 1", "content": "Запрещаются действия по возврату просроченной задолженности, которые причиняют вред жизни или здоровью должника, повреждают его имущество или оказывают на него психологическое давление. Запрещается вводить должника в заблуждение. Контекст поиска: угрозы физической расправой, угрозы испортить кредитную историю, угрозы уволить с работы, шантаж, агрессия, крик."},
    {"id": "chunk_2", "source": "Статья 7, ч. 5", "content": "Запрещается взаимодействие с должником в рабочие дни с 22 до 8 часов по местному времени, а также в выходные и нерабочие праздничные дни с 20 до 9 часов. Контекст поиска: ночное время, ночной звонок, звонок вечером, раннее утро, выходные, праздники."},
    {"id": "chunk_3", "source": "Статья 7, ч. 4", "content": "Кредитор или лицо, действующее от его имени, не вправе осуществлять взаимодействие с должником чаще двух раз в течение недели и чаще восьми раз в течение месяца. Контекст поиска: частые звонки, спам, сколько раз можно звонить, лимит звонков, постоянные смс."},
    {"id": "chunk_4", "source": "Статья 8", "content": "При взаимодействии с должником запрещается раскрывать третьим лицам сведения о наличии просроченной задолженности, а также иную информацию о должнике. Контекст поиска: третьи лица, начальник, коллеги, соседи, родственники, друзья, кредитная история, работодатель, подъезд."}
]

# ==========================================
# 🏗 3. ИНИЦИАЛИЗАЦИЯ СИСТЕМЫ (С авто-созданием БД)
# ==========================================
print("⏳ Загрузка модели и проверка базы данных...")
model = SentenceTransformer(MODEL_NAME, device="cpu")

# Подключаемся к клиенту (если папки нет, она создастся пустой)
client = chromadb.PersistentClient(path=DB_PATH)

# Пытаемся получить коллекцию. Если её нет — создаем и заполняем!
try:
    collection = client.get_collection(name="230fz_law_rich")
    print("✅ База данных найдена и загружена.")
except Exception:
    print("🆕 База данных не найдена. Создаем и индексируем...")
    collection = client.get_or_create_collection(name="230fz_law_rich")

    # Индексация
    documents = [c["content"] for c in LAW_CHUNKS]
    metadatas = [{"source": c["source"]} for c in LAW_CHUNKS]
    ids = [c["id"] for c in LAW_CHUNKS]
    embeddings = model.encode(documents, show_progress_bar=False).tolist()

    collection.add(documents=documents, metadatas=metadatas, ids=ids, embeddings=embeddings)
    print("✅ База данных создана и проиндексирована.")

# ==========================================
# 🚀 4. RAG PIPELINE (ГЛАВНАЯ ЛОГИКА)
# ==========================================
def check_compliance_rag_v2(user_text: str) -> dict:
    """
    Финальная функция проверки с Fallback-логикой.
    """
    # --- A. FALLBACK: Проверка на явные угрозы ---
    THREAT_KEYWORDS = ["приедем", "угроза", "испортим", "коллектор", "полиция", "суд", "арест"]

    if any(kw in user_text.lower() for kw in THREAT_KEYWORDS):
        # Если есть угрозы, принудительно ищем ст. 6
        fallback_query = "угрозы давление психологическое вред здоровью"
    else:
        # Иначе ищем по смыслу текста пользователя
        fallback_query = user_text

    # --- B. RETRIEVAL ---
    query_embedding = model.encode([fallback_query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=2)

    retrieved_text = results['documents'][0][0]
    article_source = results['metadatas'][0][0]['source']

    # --- C. GENERATION (Симуляция ответа LLM) ---
    print(f"🔍 Найдена норма: {article_source}")

    # Логика ответа (эмуляция работы LLM на основе контекста)
    threat_patterns = ["приедем", "угро", "испортим ки", "коллектор", "полиция", "суд", "арест", "описать имущество"]
    is_threat = any(p in user_text.lower() for p in threat_patterns)

    if is_threat:
        return {
            "status": "VIOLATION",
            "source": "Статья 6, ч. 1",
            "reason": "Фраза содержит признаки психологического давления или угрозы (даже если retrieval нашел другую статью)",
            "advice": "Удалите упоминания о приезде, силовых действиях или негативных последствиях."
        }
    elif any(kw in user_text.lower() for kw in ["23:00", "22:", "ночь", "утра"]):
        return {
            "status": "VIOLATION",
            "source": "Статья 7, ч. 5",
            "reason": "Упоминание времени взаимодействия вне разрешенного диапазона",
            "advice": "Укажите время в рамках 08:00-22:00."
        }
    else:
        return {
            "status": "COMPLIANT",
            "source": article_source,
            "reason": "Текст не содержит явных нарушений в рамках найденного контекста",
            "advice": "Текст безопасен."
        }

# ==========================================
# 🧪 5. ФИНАЛЬНЫЕ ТЕСТЫ
# ==========================================
test_cases = [
    "Ваш сосед знает про ваш долг.",
    "Оплатите сегодня или мы приедем.",
    "Звоните завтра в 9 утра, договоримся."
]

print("\n🚀 ЗАПУСК ФИНАЛЬНОГО RAG-ПАЙПЛАЙНА")
print("=" * 60)

for text in test_cases:
    print(f"\n📝 Входящее сообщение: \"{text}\"")
    result = check_compliance_rag_v2(text)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("-" * 40)

print("\n🎉 Этап 3 завершен успешно!")

✅ Библиотеки загружены.
⏳ Загрузка модели и проверка базы данных...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🆕 База данных не найдена. Создаем и индексируем...
✅ База данных создана и проиндексирована.

🚀 ЗАПУСК ФИНАЛЬНОГО RAG-ПАЙПЛАЙНА

📝 Входящее сообщение: "Ваш сосед знает про ваш долг."
🔍 Найдена норма: Статья 8
{
  "status": "COMPLIANT",
  "source": "Статья 8",
  "reason": "Текст не содержит явных нарушений в рамках найденного контекста",
  "advice": "Текст безопасен."
}
----------------------------------------

📝 Входящее сообщение: "Оплатите сегодня или мы приедем."
🔍 Найдена норма: Статья 6, ч. 1
{
  "status": "VIOLATION",
  "source": "Статья 6, ч. 1",
  "reason": "Фраза содержит признаки психологического давления или угрозы (даже если retrieval нашел другую статью)",
  "advice": "Удалите упоминания о приезде, силовых действиях или негативных последствиях."
}
----------------------------------------

📝 Входящее сообщение: "Звоните завтра в 9 утра, договоримся."
🔍 Найдена норма: Статья 7, ч. 5
{
  "status": "VIOLATION",
  "source": "Статья 7, ч. 5",
  "reason": "Упоминание времени взаи